# Task 6 — Simple Objects (count figures in a blueprint)

Count how many geometric figures appear in a 2048x2048 RGB image. Each image holds between
100 and 500 triangles, rectangles and circles, drawn with solid coloured borders.

Statement: [`../qualification/task6_Simple_Objects.md`](../qualification/task6_Simple_Objects.md)
Reasoning behind every choice here: [`task6_simple_objects.md`](./task6_simple_objects.md)

**Before running:** put the train images, `train.csv` (image name plus true count) and the test
images next to this notebook, then set the paths below. Dataset links are in the statement
(Yandex.Disk).

This task needs **no machine learning**. It is solved by classical image processing, because the
statement guarantees that each figure is a single connected component of edge pixels.

In [ ]:
TRAIN_DIR = "train_images"
TEST_DIR  = "test_images"
TRAIN_CSV = "train.csv"      # true counts, used only to calibrate two constants
OUT       = "submit.csv"

# The two numbers that decide the score. They are calibrated below, not guessed.
COLOR_TOL = 20      # how far a pixel must sit from the background to count as a stroke
MIN_AREA  = 20      # contours smaller than this are anti-aliasing speckle, not figures

In [ ]:
import numpy as np
import pandas as pd
import cv2
from pathlib import Path

print("opencv", cv2.__version__)

## The counting function

Three decisions matter, and all three come straight from the statement.

1. The background is only promised to be *distinguishable*, never to be white, so its value is
   measured per image as the most common grey level.
2. `RETR_EXTERNAL` returns only outermost outlines. A hollow shape's border produces an outer
   **and** an inner contour, so `RETR_LIST` would roughly double every count.
3. A small area filter removes anti-aliasing speckle without removing genuinely small figures.

In [ ]:
def count_figures(path, color_tol=COLOR_TOL, min_area=MIN_AREA):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Background = most common grey level in THIS image.
    bg = int(np.bincount(gray.ravel()).argmax())
    mask = (np.abs(gray.astype(np.int16) - bg) > color_tol).astype(np.uint8) * 255

    # Close 1-pixel gaps so a stroke stays one connected component.
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))

    # RETR_EXTERNAL: outermost outlines only. This is the whole task.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return sum(1 for c in contours if cv2.contourArea(c) >= min_area)

## Calibrate on the training images

The metric is relative error, so an image holding 100 figures punishes a miscount five times
harder than one holding 500. Calibration therefore reports the score, not just the mean error.

In [ ]:
FLOOR = 0.55

def score(pred, true):
    pred, true = np.asarray(pred, float), np.asarray(true, float)
    acc = 1.0 - np.minimum(1.0, np.abs(pred - true) / true)
    verdict = np.where(acc < FLOOR, 0.0, (acc - FLOOR) / (1.0 - FLOOR))
    return {"mean_accuracy": acc.mean(), "mean_verdict": verdict.mean(),
            "below_floor": int((acc < FLOOR).sum())}

train = pd.read_csv(TRAIN_CSV)
name_col = train.columns[0]
count_col = train.columns[-1]
sample = train.sample(min(40, len(train)), random_state=42)
print("calibrating on", len(sample), "images |", name_col, "->", count_col)

In [ ]:
results = []
for tol in (10, 20, 30, 40):
    for area in (5, 20, 50, 100):
        preds, trues = [], []
        for _, r in sample.iterrows():
            p = Path(TRAIN_DIR) / str(r[name_col])
            if not p.exists():
                continue
            preds.append(count_figures(p, tol, area))
            trues.append(r[count_col])
        if preds:
            s = score(preds, trues)
            results.append({"tol": tol, "area": area, **s})

res = pd.DataFrame(results).sort_values("mean_verdict", ascending=False)
print(res.head(10).to_string(index=False))

if len(res):
    COLOR_TOL = int(res.iloc[0]["tol"])
    MIN_AREA = int(res.iloc[0]["area"])
    print("chosen COLOR_TOL =", COLOR_TOL, "| MIN_AREA =", MIN_AREA)

## Estimate the runtime before launching the full job

8000 test images at 2048x2048. Time ten, extrapolate, and decide whether it fits the round
before committing twenty minutes to it.

In [ ]:
import time

test_files = sorted(Path(TEST_DIR).iterdir())
print("test images:", len(test_files))

t0 = time.time()
for p in test_files[:10]:
    count_figures(p, COLOR_TOL, MIN_AREA)
per_image = (time.time() - t0) / 10
print(f"{per_image:.3f} s per image -> {per_image * len(test_files) / 60:.1f} min for all")

In [ ]:
from tqdm.auto import tqdm

counts = [count_figures(p, COLOR_TOL, MIN_AREA) for p in tqdm(test_files)]

sub = pd.DataFrame({"id": range(len(counts)), "count": counts})
sub.to_csv(OUT, index=False)

assert len(sub) == len(test_files), f"{len(sub)} rows, expected {len(test_files)}"
assert sub["count"].between(0, 5000).all()
print(f"{OUT}: {len(sub)} rows OK")
print(sub["count"].describe())
sub.head()

## If the shipped template has no `id` column

The statement contradicts itself: the Output section asks for "a single integer in each row",
while the next sentence refers to "the order of `id` values in `submit.csv`". Copy whatever the
shipped `submit.csv` template does rather than trusting the prose.

In [ ]:
# Single-column variant, in case the template has no id column:
# pd.DataFrame({"count": counts}).to_csv(OUT, index=False, header=False)